In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# Load the ratings dataframe from the parquet tables we created
artist_ratings = pd.read_parquet('./data/mb_artist_ratings.parquet')
album_ratings = pd.read_parquet('./data/mb_album_ratings.parquet')

# Verify the loads
dataframes = {
    "Artist Ratings": artist_ratings,
    "Album Ratings": album_ratings,
}

for name, df in dataframes.items():
    print(f"✅ {name}: {df.shape[0]:,} rows loaded.")

In [ ]:
def apply_zero_anchored_bayesian(df_dict, confidence_constant=5):
    """
    Applies a zero-anchored Bayesian weighting to a dictionary of dataframes.
    Designed specifically for sparse ML features (<5% rated), forcing unrated
    and low-count items toward a baseline of 0.
    
    Parameters:
    -----------
    df_dict : dict
        Dictionary of pandas DataFrames containing 'rating' and 'rating_count'.
    confidence_constant : int or float, default 5
        The 'C' factor. How aggressively to penalize low rating counts.
        
    Returns:
    --------
    dict
        A new dictionary with DataFrames containing the engineered 'weighted_score'.
    """
    processed_dict = {}
    C = confidence_constant
    
    for name, df in df_dict.items():
        df_clean = df.copy()
        
        # Check for required columns
        if 'rating' not in df_clean.columns or 'rating_count' not in df_clean.columns:
            print(f"⚠️ Skipping '{name}': Missing required columns.")
            continue
            
        # Extract and fill nulls safely for matrix math
        v = df_clean['rating_count'].fillna(0)
        R = df_clean['rating'].fillna(0)
        
        # Apply Option 2 Formula: (R * v) / (v + C)
        # When v=0, this naturally results in 0.0
        df_clean['weighted_score'] = (R * v) / (v + C)
        
        print(f"✅ Processed '{name}': Anchored to 0.0 | Max Score Possible ~100.0 | C = {C}")
        processed_dict[name] = df_clean
        
    return processed_dict

In [ ]:
# Process the dictionary
weighted_dfs = apply_zero_anchored_bayesian(dataframes, confidence_constant=5)

# Extract your ML-ready features
artist_features = weighted_dfs["Artist Ratings"]
album_features = weighted_dfs["Album Ratings"]

In [ ]:
# Scale the rating feature down so it shares the exact same weight as a binary country match
artist_features['weighted_score_norm'] = artist_features['weighted_score'] / 100.0
album_features['weighted_score_norm'] = album_features['weighted_score'] / 100.0

In [ ]:
# Set up the plotting style for clean, readable visuals
sns.set_theme(style="whitegrid")

# Create a 2x2 grid of plots
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Extract dataframes from your processed dictionary for convenience
artist_df = weighted_dfs["Artist Ratings"]
album_df = weighted_dfs["Album Ratings"]

# ----------------------------------------------------------------------
# ROW 1: OVERALL DISTRIBUTIONS (Log Scale to handle 95% Zeros)
# ----------------------------------------------------------------------

# Top Left: All Artists
sns.histplot(data=artist_df, x='weighted_score_norm', bins=50, ax=axes[0, 0], color='#4A90E2', kde=False)
axes[0, 0].set_yscale('log')  # Crucial for seeing the sparse tail
axes[0, 0].set_title('Overall Artist Scores (All Data - Log Scale)', fontsize=13, weight='bold', pad=10)
axes[0, 0].set_xlabel('Weighted Score', fontsize=11)
axes[0, 0].set_ylabel('Count (Log Scale)', fontsize=11)

# Top Right: All Albums
sns.histplot(data=album_df, x='weighted_score_norm', bins=50, ax=axes[0, 1], color='#E056FD', kde=False)
axes[0, 1].set_yscale('log')  # Crucial for seeing the sparse tail
axes[0, 1].set_title('Overall Album Scores (All Data - Log Scale)', fontsize=13, weight='bold', pad=10)
axes[0, 1].set_xlabel('Weighted Score', fontsize=11)
axes[0, 1].set_ylabel('Count (Log Scale)', fontsize=11)


# ----------------------------------------------------------------------
# ROW 2: ACTIVE SIGNALS ONLY (Filtering out the 0.0 unrated rows)
# ----------------------------------------------------------------------

# Filter for rows that actually received ratings
active_artists = artist_df[artist_df['weighted_score_norm'] > 0]
active_albums = album_df[album_df['weighted_score_norm'] > 0]

# Bottom Left: Rated Artists Only
sns.histplot(data=active_artists, x='weighted_score_norm', bins=40, ax=axes[1, 0], color='#10AC84', kde=True)
axes[1, 0].set_title(f'Distribution of Rated Artists Only (N={len(active_artists)})', fontsize=13, weight='bold', pad=10)
axes[1, 0].set_xlabel('Weighted Score (> 0)', fontsize=11)
axes[1, 0].set_ylabel('Count (Linear Scale)', fontsize=11)

# Bottom Right: Rated Albums Only
sns.histplot(data=active_albums, x='weighted_score_norm', bins=40, ax=axes[1, 1], color='#FF6B6B', kde=True)
axes[1, 1].set_title(f'Distribution of Rated Albums Only (N={len(active_albums)})', fontsize=13, weight='bold', pad=10)
axes[1, 1].set_xlabel('Weighted Score (> 0)', fontsize=11)
axes[1, 1].set_ylabel('Count (Linear Scale)', fontsize=11)


# Adjust layout to prevent any text/label overlapping or truncation
plt.tight_layout()

# Render the plots in your notebook
plt.show()

In [ ]:
import os
import pickle
from scipy.sparse import csr_matrix, save_npz

# 1. Ensure the target directory exists
os.makedirs('data/features', exist_ok=True)

# 2. Load the index mappings exported by the tags notebook
print("Loading master row index configurations...")
with open('data/features/artist_ids.pkl', 'rb') as f:
    unique_artist_ids = pickle.load(f)

with open('data/features/album_ids.pkl', 'rb') as f:
    unique_album_ids = pickle.load(f)

# 3. Align and export Artist Ratings
print("Aligning and exporting Artist Ratings...")
aligned_artist_ratings = artist_features.reindex(unique_artist_ids)['weighted_score_norm'].fillna(0.0).values
X_artist_ratings_sparse = csr_matrix(aligned_artist_ratings.astype('float32')).T
save_npz('data/features/artist_ratings_matrix.npz', X_artist_ratings_sparse)

# 4. Align and export Album Ratings
print("Aligning and exporting Album Ratings...")
aligned_album_ratings = album_features.reindex(unique_album_ids)['weighted_score_norm'].fillna(0.0).values
X_album_ratings_sparse = csr_matrix(aligned_album_ratings.astype('float32')).T
save_npz('data/features/album_ratings_matrix.npz', X_album_ratings_sparse)

print("🎉 Ratings successfully exported as sparse vectors to data/features/")